In [1]:
# Importo la libreria pandas
import pandas as pd

# Leggo il file compresso direttamente (pandas legge i .gz senza estrarli)
df = pd.read_csv("dataraw/migr_resfirst__custom_22885892_linear.csv.gz")

# Guardo le prime 5 righe
df.head()

,DATAFLOW,LAST UPDATE,freq,reason,citizen,duration,unit,geo,TIME_PERIOD,OBS_VALUE,OBS_FLAG,CONF_STATUS
0,ESTAT:MIGR_RESFIRST(1.0),11/09/26 11:00:00,Annual,Education reasons,Albania,Total,Person,Austria,2008,39,NaN,NaN
1,ESTAT:MIGR_RESFIRST(1.0),11/09/26 11:00:00,Annual,Education reasons,Albania,Total,Person,Austria,2009,48,NaN,NaN
2,ESTAT:MIGR_RESFIRST(1.0),11/09/26 11:00:00,Annual,Education reasons,Albania,Total,Person,Austria,2010,48,NaN,NaN
3,ESTAT:MIGR_RESFIRST(1.0),11/09/26 11:00:00,Annual,Education reasons,Albania,Total,Person,Austria,2011,69,NaN,NaN
4,ESTAT:MIGR_RESFIRST(1.0),11/09/26 11:00:00,Annual,Education reasons,Albania,Total,Person,Austria,2012,86,NaN,NaN


In [2]:
# Quante righe e quante colonne ha la tabella?
print("Righe e colonne:", df.shape)

# Quali sono i nomi delle colonne?
print(df.columns.tolist())

Righe e colonne: (2870, 12)
['DATAFLOW', 'LAST UPDATE', 'freq', 'reason', 'citizen', 'duration', 'unit', 'geo', 'TIME_PERIOD', 'OBS_VALUE', 'OBS_FLAG', 'CONF_STATUS']


In [3]:
# Quanti valori diversi ci sono in ogni colonna?
df.nunique()

DATAFLOW         1
LAST UPDATE      1
freq             1
reason           5
citizen          1
duration         1
unit             1
geo             35
TIME_PERIOD     18
OBS_VALUE      767
OBS_FLAG         3
CONF_STATUS      0
dtype: int64

In [4]:
# Tengo solo le 5 colonne utili
df = df[["geo", "TIME_PERIOD", "reason", "OBS_VALUE", "OBS_FLAG"]]

# Rinomino le colonne in italiano, più chiare
df = df.rename(columns={
    "geo": "paese",
    "TIME_PERIOD": "anno",
    "reason": "motivo",
    "OBS_VALUE": "permessi",
    "OBS_FLAG": "flag"
})

df.head()

,paese,anno,motivo,permessi,flag
0,Austria,2008,Education reasons,39,NaN
1,Austria,2009,Education reasons,48,NaN
2,Austria,2010,Education reasons,48,NaN
3,Austria,2011,Education reasons,69,NaN
4,Austria,2012,Education reasons,86,NaN


In [5]:
# Traduco i motivi in italiano
traduzione_motivi = {
    "Education reasons": "Studio",
    "Employment reasons": "Lavoro",
    "Family reasons": "Famiglia",
    "Other reason": "Altro",
    "Total": "Totale"
}
df["motivo"] = df["motivo"].replace(traduzione_motivi)

# Controllo: quante righe per ogni motivo?
df["motivo"].value_counts()

motivo
Studio      575
Lavoro      575
Famiglia    575
Altro       573
Totale      572
Name: count, dtype: int64

In [6]:
# Righe prima della pulizia
print("Righe prima:", len(df))

# Tolgo gli aggregati UE (non sono paesi, sono totali)
df = df[~df["paese"].str.contains("European Union")]

# Righe dopo la pulizia
print("Righe dopo:", len(df))
print("Numero di paesi:", df["paese"].nunique())

Righe prima: 2870
Righe dopo: 2774
Numero di paesi: 33


In [7]:
# Quante righe per ogni flag? (dropna=False conta anche le righe senza flag)
df["flag"].value_counts(dropna=False)

flag
NaN    2710
u        33
d        18
b        13
Name: count, dtype: int64

In [8]:
# Salvo la tabella pulita nella cartella dataclean
df.to_csv("dataclean/permessi_primi_albanesi.csv", index=False)

print("File salvato!", len(df), "righe,", len(df.columns), "colonne")

File salvato! 2774 righe, 5 colonne
